# 05. Context Windows, Attention Complexity & KV Caching

The **Context Window** defines the maximum number of tokens an LLM can attend to in a single forward pass.

### The $O(N^2)$ Complexity Problem:
Standard full self-attention computes an $N \times N$ attention matrix. Doubling context length from 4K to 128K scales memory and compute by $32^2 = 1,024\times$!

### FlashAttention (Dao et al., 2022):
Re-computes attention on-chip in fast GPU SRAM using tiling, reducing memory reads and achieving exact attention with $O(N)$ memory and $2\times - 4\times$ wall-clock speedups.

In [ ]:
import numpy as np

def estimate_kv_cache_gb(n_layers=32, n_kv_heads=8, head_dim=128, seq_len=4096, batch_size=1, dtype_bytes=2):
    # KV Cache bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * dtype_bytes
    total_bytes = 2 * n_layers * n_kv_heads * head_dim * seq_len * batch_size * dtype_bytes
    return total_bytes / (1024 ** 3)

for sl in [2048, 8192, 32768, 131072]:
    gb = estimate_kv_cache_gb(seq_len=sl)
    print(f"Context Length: {sl:6d} tokens -> KV Cache Memory: {gb:.2f} GB")

## 🎯 Exercises & Practice
1. Explain the "Lost in the Middle" phenomenon in long-context models (Liu et al., 2023).

In [ ]:
# Exercise Solution
# Answer: Models retrieve facts best when positioned near the very beginning or end of the context window,
# but performance degrades severely when information is buried in the middle of long contexts!